# Connectome as a Valence Estimator (perception-only)

**Question.** The larva olfactory circuit (ORN→KC→MBON) computes *valence* — "am I
moving up-gradient?" — not motor commands. So instead of end-to-end steering, we use
the real connectome as a **valence estimator** and ask the cleanest question it can
actually answer:

> Given only the bilateral sensor stream over time, can the connectome reconstruct
> **dlog** (the temporal change of log-concentration)?

dlog is computable by the circuit (a temporal derivative of its own inputs); absolute
distance / global bearing are not — so dlog is the biologically honest target.

**Why this is easy + rigorous.** Supervised regression (dense gradient, backprop) — no
sparse-reward RL. We compare the connectome against baselines that *cannot* integrate
over time, so beating them proves the recurrence does real temporal integration.

## 1. Setup

In [ ]:
import os, sys, subprocess
if os.path.isdir('/content'):
    # --- Colab: clone the repo (no Drive mount needed) ---
    REPO_URL = 'https://github.com/InHyunseo/Brain-inspired-OSL.git'
    REPO_DIR = '/content/2d-osl'
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call(['git','clone','--depth','1', REPO_URL, REPO_DIR])
else:
    # --- Local: find the repo root (dir containing src/) ---
    REPO_DIR=os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(REPO_DIR,'src')) and REPO_DIR!=os.path.dirname(REPO_DIR):
        REPO_DIR=os.path.dirname(REPO_DIR)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0,REPO_DIR)
print('repo:', REPO_DIR)

import numpy as np, torch
import matplotlib.pyplot as plt
from src.brain.valence_connectome import ValenceConnectome
from src.brain.train_valence import make_dataset, to_tensors, baseline_scores, train, masked_mse
print('imports OK | torch', torch.__version__)

## 2. Config (edit here)

In [ ]:
# dataset
N_EPISODES   = 300      # random-policy rollouts
EP_LEN       = 120      # steps per episode
SEQ_T        = 120      # padded sequence length
DATA_SEED    = 0
ENV_KW = dict(success_radius_mm=7.5, gaussian_sigma_mm=30.0, episode_seconds=120.0,
              sensor_spacing_mm=0.15)

# connectome
INNER_STEPS  = 4        # message-passing steps per env step
N_OUT_NODES  = 8

# training
EPOCHS       = 40
LR           = 1e-2
BATCH        = 64
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## 3. Build dataset (sensor sequence → dlog GT)

Random-policy rollouts give diverse sensor patterns. Input = (c_left, c_right) per
step; target = dlog (already computed by the env).

In [ ]:
print('generating dataset ...')
S, D = make_dataset(n_episodes=N_EPISODES, ep_len=EP_LEN, seed=DATA_SEED, env_kw=ENV_KW)
Sx, Dy, mask = to_tensors(S, D, T=SEQ_T, device=DEVICE)
print('episodes:', Sx.shape[0], '| padded T:', Sx.shape[1], '| valid steps:', int(mask.sum().item()))
print('dlog target — mean %.4f  std %.4f' % (Dy[mask.bool()].mean().item(), Dy[mask.bool()].std().item()))

## 4. Baselines (no temporal memory)

- **const-0**: predict 0 always → MSE = variance of dlog (trivial floor).
- **instantaneous-linear**: best linear map of (c_L, c_R, c_L−c_R) → can a purely
  *spatial* readout fake the temporal derivative? (closed-form least squares)

In [ ]:
bl = baseline_scores(Sx.cpu(), Dy.cpu(), mask.cpu())
print('const-0 MSE          : %.5f   (dlog variance, trivial floor)' % bl['const0_mse'])
print('instantaneous-linear : %.5f   (spatial-only, no memory)' % bl['instantaneous_mse'])

## 5. Train the connectome valence regressor

In [ ]:
model = ValenceConnectome(n_output_nodes=N_OUT_NODES, inner_steps=INNER_STEPS).to(DEVICE)
print('connectome:', model.describe())
res = train(model, Sx, Dy, mask, epochs=EPOCHS, lr=LR, batch=BATCH, seed=0, verbose=True)
print('\nbest val MSE:', round(res['best_val_mse'],5))

## 6. Result — does the connectome beat the memory-less baselines?

In [ ]:
conn = res['best_val_mse']
names = ['const-0','instantaneous\n(spatial)','connectome\n(temporal)']
vals  = [bl['const0_mse'], bl['instantaneous_mse'], conn]
fig, ax = plt.subplots(figsize=(5.5,4))
bars = ax.bar(names, vals, color=['gray','tab:orange','tab:green'])
ax.set_ylabel('validation MSE on dlog (lower = better)')
ax.set_title('dlog reconstruction: connectome vs memory-less baselines')
for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2, v, f'{v:.4f}', ha='center', va='bottom')
ax.grid(axis='y',alpha=.3); plt.tight_layout(); plt.show()

print('connectome beats instantaneous baseline:', conn < bl['instantaneous_mse'])
print('relative improvement vs instantaneous: %.1f%%' % (100*(bl['instantaneous_mse']-conn)/bl['instantaneous_mse']))

## 7. Qualitative — predicted vs true dlog on a held-out episode

In [ ]:
model.eval()
with torch.no_grad():
    i = 0  # episode index to view
    pred = model(Sx[i:i+1])[0,:,0].cpu().numpy()
T = int(mask[i].sum().item())
true = Dy[i,:T,0].cpu().numpy(); pred = pred[:T]
fig, ax = plt.subplots(figsize=(9,3.2))
ax.plot(true, label='true dlog', lw=2)
ax.plot(pred, label='connectome prediction', lw=1.5, alpha=.8)
ax.axhline(0, color='k', lw=.5)
ax.set_xlabel('env step'); ax.set_ylabel('dlog'); ax.legend()
ax.set_title('Predicted vs true dlog over one episode')
plt.tight_layout(); plt.show()
print('correlation:', round(float(np.corrcoef(true,pred)[0,1]),3))

## 8. (optional) Save the trained connectome

In [ ]:
from pathlib import Path
OUT = Path('runs')/'valence_connectome'; OUT.mkdir(parents=True, exist_ok=True)
torch.save({'state_dict':model.state_dict(),'describe':model.describe(),
            'best_val_mse':res['best_val_mse'],'baselines':bl,
            'config':dict(inner_steps=INNER_STEPS,n_out=N_OUT_NODES,env_kw=ENV_KW)},
           OUT/'valence_connectome.pt')
print('saved', OUT/'valence_connectome.pt')